In [1]:
# 1. Imports
import sys
sys.path.append("..")

import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score, GridSearchCV
from src.preprocessing import preprocess

# 2. Load raw data
raw_train = pd.read_csv("../data/raw/train.csv")
raw_test = pd.read_csv("../data/raw/test.csv")

# 3. Compute the three "learned from train only" lookup tables, BEFORE any preprocessing call
temp_title = raw_train["Name"].str.extract(r",\s*([^.]+)\.")[0]
temp_title = temp_title.where(temp_title.isin(["Mr", "Miss", "Mrs", "Master"]), "Rare")
age_medians_by_title = raw_train["Age"].groupby(temp_title).median()

fare_medians_by_pclass = raw_train.groupby("Pclass")["Fare"].median()

combined_ticket_counts = pd.concat([raw_train["Ticket"], raw_test["Ticket"]]).value_counts()

# 4. Preprocess both train and test, using those same fixed lookup tables
train_processed = preprocess(raw_train, age_medians_by_title, combined_ticket_counts, fare_medians_by_pclass)
test_processed = preprocess(raw_test, age_medians_by_title, combined_ticket_counts, fare_medians_by_pclass)

# 5. Build X, y
feature_cols = [
    "IsFemale", "Pclass", "Age", "Fare", "FamilySize", "HasCabin",
    "Title_Master", "Title_Miss", "Title_Mr", "Title_Mrs", "Title_Rare",
    "Embarked_C", "Embarked_Q", "Embarked_S", "TicketGroupSize"
]
X = train_processed[feature_cols]
y = train_processed["Survived"]
X_test_final = test_processed[feature_cols]

# 6. Sanity check — MUST show 0 and 0 before doing anything else
print(X.isna().sum().sum(), "missing values in X")
print(X_test_final.isna().sum().sum(), "missing values in X_test_final")

0 missing values in X
0 missing values in X_test_final


In [3]:
param_grid = {"max_depth": [4, 6, 8, 10], "n_estimators": [50, 100, 200]}

inner_cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=1)
outer_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

grid_search = GridSearchCV(RandomForestClassifier(random_state=42), param_grid, cv=inner_cv)

nested_scores = cross_val_score(grid_search, X, y, cv=outer_cv)
print(nested_scores)
print(f"nested mean={nested_scores.mean()*100:.2f}%, std={nested_scores.std()*100:.2f}")

[0.84357542 0.83146067 0.83146067 0.84269663 0.84831461]
nested mean=83.95%, std=0.68
